# 25 · Self-heterodyne linewidth: short delays, flicker noise and polarization fading

The classic measurement of [24](24-laser-linewidth-self-heterodyne.ipynb) needs a delay much longer
than the laser's coherence time.  This notebook covers what happens when
that is not possible or not enough:

1. the **coherent regime** of a short delay, where the Lorentzian width is
   meaningless but the phase-based estimators still work;
2. **flicker (1/f) FM noise**, which removes the notion of one linewidth;
3. **long-term stability**: the Allan deviation from the interferometer as
   a frequency discriminator;
4. **polarization fading** and a dual-polarization receiver.

The setup, the forward model (`analysis.dsh_beat`) and the three estimators
are introduced in [24](24-laser-linewidth-self-heterodyne.ipynb).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from commkit import analysis, plotting
from commkit.backend import is_cupy_available, to_device
from commkit.impairments import apply_awgn, generate_phase_noise

if is_cupy_available():
    import cupy as xp  # arrays (and all analysis) on the GPU
else:
    xp = np  # CPU fallback - everything below is backend-agnostic


def cpu(a):
    """Device->host helper for hand-made matplotlib overlays."""
    return to_device(a, "cpu")


DEVICE = "cpu" if xp is np else "gpu"

plotting.apply_default_theme()

In [ ]:
FS = 500e6  # ADC sampling rate [Hz]
N = 1 << 21  # record length (≈ 4.2 ms)
DNU_TRUE = 200e3  # laser linewidth to be recovered [Hz]
F_AOM = 80e6  # AOM shift [Hz]
SNR_DB = 25  # beat SNR in the full Nyquist band

## 1. When the spool cannot be long: the coherent regime

The incoherent condition `τ_d ≫ τ_c` scales inversely with the linewidth:
a **10 kHz laser needs τ_d ≫ 32 µs -> hundreds of km** once a safety factor
is applied - impractical (0.2 dB/km loss, and every meter of fiber is a
microphone).  For narrow-linewidth lasers the modern approach is therefore
a *short* delay analyzed with the phase-based estimators, which do not
care about the regime.  The 200 kHz laser of [24](24-laser-linewidth-self-heterodyne.ipynb), now with a 1 km spool
(`τ_d/τ_c ≈ 3` -> coherent):

* the beat spectrum shows the tell-tale carrier spike + fringes spaced
  `1/τ_d` - a Lorentzian fit to *this* is meaningless: most of the power
  sits in the un-broadened spike whose width is the Welch **resolution**,
  and the estimator warns (`coherence_factor < 6`);
* `DshFmPsd` (and `DshIncrement`) on the *same capture* still recover Δν - for
  `f ≪ 1/τ_d` the interferometer is a frequency discriminator with known
  gain `(2πfτ_d)²`, regime-independent.

In [ ]:
M_SHORT = int(round(4.9e-6 * FS))  # 1 km spool
TAU_SHORT = M_SHORT / FS

phi_short = generate_phase_noise(
    num_samples=N + M_SHORT, sampling_rate=FS, linewidth=DNU_TRUE, rng=44, device=DEVICE
)
z_short, _ = analysis.dsh_beat(
    phi_short, sampling_rate=FS, delay=TAU_SHORT, f_shift=F_AOM
)
z_short = apply_awgn(z_short, sps=1, esn0_db=SNR_DB, rng=3)

plotting.plot_psd(
    z_short,
    sampling_rate=FS,
    nperseg=1 << 15,
    xlim=((F_AOM - 3e6) / 1e6, (F_AOM + 3e6) / 1e6),
    title=f"Coherent-regime beat: spike + fringes every 1/τ_d = {1e-6 / TAU_SHORT:.3f} MHz",
)
plt.show()

res_bad = analysis.estimate_linewidth(
    z_short, analysis.DshLorentzian(delay=TAU_SHORT, nperseg=1 << 15), sampling_rate=FS
)
res_good = analysis.estimate_linewidth(
    z_short,
    analysis.DshFmPsd(delay=TAU_SHORT, f_shift=F_AOM),
    sampling_rate=FS,
)
print(
    f"coherent-regime 'lorentzian' answer: {res_bad.value / 1e3:.1f} kHz "
    f"(truth {DNU_TRUE / 1e3:.0f} kHz) - coherence factor "
    f"{res_bad.coherence_factor:.2f} < 6, do not trust it."
)
print(f"fm_psd on the same capture         : {res_good.value / 1e3:.1f} kHz")

## 2. What 1/f (flicker) FM noise does

Real lasers are never purely white-FM: below some corner frequency the FM
PSD rises as `~1/f`.  `generate_phase_noise` models both terms of the
standard power-law FM noise in one call,

$$S_f(f) = \underbrace{\Delta\nu/\pi}_{\texttt{linewidth}} +
           \underbrace{h_{-1}/f}_{\texttt{flicker}},$$

so the laser below is the same white-FM laser plus a flicker term `h₋₁`
chosen to cross the white plateau at ≈ 63 kHz (`flicker_f_min` caps the
1/f divergence at the lowest shaped frequency).  Consequences:

* there is **no unique "linewidth"** anymore - the measured width grows
  with observation time;
* the -3 dB width inflates (Gaussian core), while the far wings - and the
  white-FM plateau in the FM PSD - still reflect the intrinsic Lorentzian
  part;
* on the FM-PSD plot the flicker shows up directly as the low-frequency
  rise, which is *the* most honest way to report it.

The fit carries an `h₋₁/f` term, so the flicker does not bias the white-FM
level.  Measuring `h₋₁` itself needs bins inside the flicker region: the
default segments start at about `1/τ_d ≈ 204 kHz`, above this 63 kHz
corner, so they report `h₋₁` low.  Segments of 2^15 samples (15 kHz bins)
resolve the fringes *and* the corner, and recover it.  A naive median over
the first lobe, by contrast, reads the flicker-inflated level.

In [ ]:
H_M1 = 4e9
phi_mix = generate_phase_noise(
    num_samples=N + M_SHORT,
    sampling_rate=FS,
    linewidth=DNU_TRUE,
    flicker=H_M1,
    flicker_f_min=1e3,
    rng=45,
    device=DEVICE,
)
z_mix, _ = analysis.dsh_beat(phi_mix, sampling_rate=FS, delay=TAU_SHORT, f_shift=F_AOM)
z_mix = apply_awgn(z_mix, sps=1, esn0_db=SNR_DB, rng=4)

res_mix = analysis.estimate_linewidth(
    z_mix, analysis.DshFmPsd(delay=TAU_SHORT, f_shift=F_AOM), sampling_rate=FS
)
# The naive read: π times the median deconvolved PSD inside the first lobe.
# The default segments are as long as τ_d, one bin wider than that lobe, so
# take it from segments that resolve the fringes.
res_lobes = analysis.estimate_linewidth(
    z_mix,
    analysis.DshFmPsd(delay=TAU_SHORT, f_shift=F_AOM, nperseg=1 << 15),
    sampling_rate=FS,
)
first_lobe = (res_lobes.f > 0) & (res_lobes.f <= 1.0 / TAU_SHORT)
naive = np.pi * np.nanmedian(res_lobes.S_f[first_lobe])

fig, ax = plotting.plot_frequency_noise_psd(
    res_mix,
    title="Flicker rises above the white plateau at low f - report the PSD, not one number",
)
fline = np.geomspace(2e5, 2e7, 50)
ax.loglog(fline, H_M1 / fline, "--", label=r"$h_{-1}/f$ injected flicker")
ax.axhline(DNU_TRUE / np.pi, color="C3", ls=":", label=r"White plateau $\Delta\nu/\pi$")
ax.legend()
plt.show()

print(f"naive first-lobe median : Δν = {naive / 1e3:.0f} kHz (flicker-inflated)")
print(f"fitted white FM         : Δν = {res_mix.value / 1e3:.0f} kHz")
print(
    f"fitted flicker h₋₁      : {res_lobes.flicker:.2e} Hz² with 2^15 segments, "
    f"{res_mix.flicker:.2e} with the default (injected {H_M1:.0e})"
)
print(f"truth (white-FM part)   : Δν = {DNU_TRUE / 1e3:.0f} kHz")

## 3. Long-term stability - Allan deviation from the discriminator

Below the first notch the interferometer is a frequency discriminator:
`Δφ(t)/(2πτ_d)` is the laser's instantaneous frequency averaged over the
τ_d window.  Feeding it to `analysis.allan_deviation` gives the laser
frequency stability σ_y(τ) - **valid for τ ≫ τ_d**.

The short-delay capture is used here deliberately: its validity floor
(`≈ 5·τ_d = 25 µs`) leaves two decades of usable τ in this record, whereas
the 25 km spool would push the floor to 0.6 ms - with only 4.2 ms of data.
A wider Allan range is one more genuine advantage of short delays.

For pure white FM the expected curve is `σ_y(τ) = √(Δν/(2πτ))` - the
`τ^{-1/2}` guide line of the plot.  A real laser flattens (flicker) and
turns up (random-walk / temperature drift) at long τ; extend the record,
not the sampling rate, to see that region.

In [ ]:
dphi_short, _ = analysis.dsh_phase(z_short, sampling_rate=FS, f_shift=F_AOM)
df_disc = dphi_short / (2.0 * np.pi * TAU_SHORT)  # discriminator output [Hz]
allan = analysis.allan_deviation(df_disc, sampling_rate=FS, num_taus=40)

fig, ax = plotting.plot_allan_deviation(allan)
tau_valid = allan.tau_s[allan.tau_s > 5 * TAU_SHORT]
ax.loglog(
    tau_valid,
    np.sqrt(DNU_TRUE / (2.0 * np.pi * tau_valid)),
    "--",
    color="C3",
    label=r"White-FM theory $\sqrt{\Delta\nu/2\pi\tau}$",
)
ax.axvspan(
    allan.tau_s[0],
    5 * TAU_SHORT,
    color="gray",
    alpha=0.4,
    label=r"$\tau \lesssim 5\tau_d$: invalid",
)
ax.legend()
plt.show()

## 4. Polarization fading and a dual-polarization receiver

Only the part of one arm's field that is *co-polarized* with the other arm
beats.  The delay spool's birefringence drifts with temperature and
vibration, so the two arms' states of polarization (SOPs) wander relative
to each other, and a single-polarization receiver sees the beat amplitude
fade - to zero when the SOPs become orthogonal.  During a fade the beat
phase is pure noise: unwrapping slips, and the linewidth estimate is ruined.
(A polarization controller in one arm helps until the spool drifts again.)

A **dual-polarization coherent receiver** records the beat in two orthogonal
polarizations,

$$z_p(t) = a_p(t)\,s(t) + n_p(t), \qquad p = x, y,$$

where `s(t)` is the beat both outputs share, `a = (a_x, a_y)` the slowly
drifting Jones vector it arrives in, and `n_p` independent receiver noise.
The total beat power `|a_x|² + |a_y|²` does not fade - only its split
between the two outputs changes.  The best estimate of the phase of `s`
combines the outputs by **maximum-ratio combining**,

$$y = \frac{a^H z}{\|a\|} = \|a\|\,s + \text{noise},$$

each output weighted by its own amplitude and co-phased.  `a` is not known,
but the coherency matrix of the outputs reveals it:
`R = ⟨z zᴴ⟩ = a aᴴ + σ² I`, so its dominant eigenvector is `a/‖a‖` (up to a
phase).  `combine_polarizations=True` estimates `R` over ≈ 1 µs blocks,
takes that eigenvector per block, aligns the phase of consecutive
eigenvectors (*parallel transport*, so the weights stay continuous through
a fade), interpolates them to the sample rate and forms `y`.

One subtlety: when the SOP traces a loop on the Poincaré sphere, `y` picks
up the geometric (Pancharatnam) phase, half the enclosed solid angle.  It
varies at the SOP's own timescale (Hz to kHz), far below the frequencies
the linewidth fit uses.

Below, the 1 km beat arrives in a SOP that swings through a full fade of
each output every few hundred µs.

In [ ]:
t = np.arange(N) / FS
alpha = np.pi / 2 * np.sin(2 * np.pi * 2e3 * t) ** 2  # x fades, then y
beta = 2 * np.pi * 3.4e3 * t
jones = xp.asarray(np.stack([np.cos(alpha), np.sin(alpha) * np.exp(1j * beta)]))
phi_pol = generate_phase_noise(
    num_samples=N + M_SHORT, sampling_rate=FS, linewidth=DNU_TRUE, rng=46, device=DEVICE
)
z_pol, _ = analysis.dsh_beat(phi_pol, sampling_rate=FS, delay=TAU_SHORT, f_shift=F_AOM)
z_pol = apply_awgn(jones * z_pol, sps=1, esn0_db=SNR_DB, rng=5)  # (2, N)

one_pol = analysis.estimate_linewidth(
    z_pol[0], analysis.DshFmPsd(delay=TAU_SHORT, f_shift=F_AOM), sampling_rate=FS
)
combined = analysis.estimate_linewidth(
    z_pol,
    analysis.DshFmPsd(delay=TAU_SHORT, f_shift=F_AOM, combine_polarizations=True),
    sampling_rate=FS,
)
print(
    f"x output alone        : Δν = {one_pol.value / 1e3:8.0f} kHz (fades are pure noise)"
)
print(
    f"both outputs combined : Δν = {combined.value / 1e3:8.1f} kHz (truth {DNU_TRUE / 1e3:.0f} kHz)"
)

**Summary**

* Short delays (`τ_d ≲ τ_c`) put the beat in the coherent regime: drop the
  Lorentzian width and use `DshFmPsd` / `DshIncrement`, which do not care.
* With flicker FM noise there is no single linewidth: report the FM-noise
  PSD; the fit separates the white plateau from `h₋₁/f`.
* Below the first notch the interferometer is a frequency discriminator;
  its Allan deviation is valid for `τ ≫ τ_d`.
* A dual-polarization receiver with `combine_polarizations=True` survives
  the fades of the spool's polarization drift.

**Next:** [26 · Self-homodyne IQ linewidth measurement](26-laser-linewidth-self-homodyne-iq.ipynb).